# Task4 — OOF class-bias adjustment

Run All ใช้ prediction caches จาก Train/validation เท่านั้น ไม่ฝึก backbone ไม่อ่าน Test และไม่สร้าง submission.

ปรับ `argmax(log(p)+bias)` ของ ensemble compound/sampler 50/50 เดิม. ใช้4 folds fit bias และวัดบนfoldที่เหลือ หมุนครบ5รอบ. นี่คือ cross-fitting ของ **bias เท่านั้น** ไม่ใช่ nested CV ของ pipeline ทั้งหมด: โมเดลที่สร้าง OOF ของ calibration folds อาจเคยฝึกบนแถวใน evaluation fold และ OOF ชุดนี้เคยใช้เลือกโมเดลแล้ว จึงเป็น exploratory estimate. ห้ามอ้างว่าเป็น independent/unbiased performance.

Fit bias บน original OOF เท่านั้น; stress development ใช้ตรวจผลกระทบ ไม่ใช้เลือก bias. ไม่เปิด final audit ซ้ำเพื่อจูน. Final bias ที่ fit บน OOF ทั้งหมดแยกจากค่าที่ใช้ประเมิน และไม่ถูกนำไปใช้ Test อัตโนมัติ.

## 1. Settings

In [1]:
from pathlib import Path
import sys,json,hashlib
import numpy as np
import pandas as pd
ROOT=next(p for b in (Path.cwd(),*Path.cwd().parents) for p in (b,b/'CPE342-Kaggle') if (p/'Task4/models/image_compound_followup.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models.image_compound_followup import probabilities
from Task4.models.image_compound_transforms import CONDITIONS
from IPython.display import display
SOURCE=ROOT/'Task4/models/results/compound_supervised_v1'
SAMPLER=ROOT/'Task4/models/results/heavy_balance_v1/heavy_sampler18'
OUT=ROOT/'Task4/models/results/class_bias_v1'
GRID=np.linspace(-.4,.4,17)
PASSES=3
PENALTY=.001
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()


## 2. Load matched OOF predictions

In [2]:
parts=[];sources={}
for fold in range(1,6):
    paths=[SOURCE/'compound_supervised6'/f'fold{fold}'/'oof.parquet',SAMPLER/f'fold{fold}'/'oof.parquet']
    a,b=[pd.read_parquet(p) for p in paths]
    assert a.id.tolist()==b.id.tolist() and np.array_equal(a.true,b.true) and np.array_equal(a.row_index,b.row_index)
    p=(probabilities(a,'tta')+probabilities(b,'tta'))/2
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5)
    f=a[['id','row_index','true']].copy();f['fold']=fold
    for j in range(5):f[f'p{j}']=p[:,j]
    parts.append(f)
    sources.update({str(path.relative_to(ROOT)):sha(path) for path in paths})
oof=pd.concat(parts,ignore_index=True)
assert oof.id.is_unique and oof.row_index.is_unique and len(oof)==31546
logp=np.log(np.clip(oof[[f'p{i}' for i in range(5)]].to_numpy(),1e-12,1))
y=oof.true.to_numpy(int);fold_ids=oof.fold.to_numpy(int)
assert set(y)==set(range(5))
print('OOF rows:',len(oof),'classes:',np.bincount(y))


OOF rows: 31546 classes: [12429  4407  5596  5926  3188]


## 3. Bounded coordinate search — Macro F1 with a small bias penalty

In [3]:
def macro_f1(y,pred):
    cm=np.bincount(5*y+pred,minlength=25).reshape(5,5)
    denom=cm.sum(0)+cm.sum(1)
    return float(np.divide(2*np.diag(cm),denom,out=np.zeros(5,dtype=float),where=denom>0).mean())

def fit_bias(lp,labels):
    # Class0 anchors additive invariance. Grid/order/penalty fixed before running.
    bias=np.zeros(5)
    def objective(b):return macro_f1(labels,(lp+b).argmax(1))-PENALTY*np.abs(b).mean()
    best=objective(bias)
    for _ in range(PASSES):
        improved=False
        for c in range(1,5):
            chosen=bias.copy()
            for value in sorted(GRID,key=lambda x:(abs(x),x)):
                candidate=bias.copy();candidate[c]=value
                score=objective(candidate)
                if score>best+1e-12:best=score;chosen=candidate;improved=True
            bias=chosen
        if not improved:break
    assert bias[0]==0 and np.isfinite(bias).all()
    return bias


## 4. Cross-fit biases and compare original OOF

In [4]:
records=[];biases={};adjusted=np.empty(len(y),dtype=int)
for fold in range(1,6):
    fit=fold_ids!=fold;valid=~fit
    bias=fit_bias(logp[fit],y[fit]);biases[fold]=bias
    baseline=logp[valid].argmax(1);pred=(logp[valid]+bias).argmax(1);adjusted[valid]=pred
    before=macro_f1(y[valid],baseline);after=macro_f1(y[valid],pred)
    records.append({'fold':fold,'baseline':before,'adjusted':after,'delta':after-before,'changed':int((pred!=baseline).sum())})
fold_results=pd.DataFrame(records)
display(fold_results)
display(pd.DataFrame(biases,index=[f'class{i}' for i in range(5)]).T)
print('Pooled original F1:',macro_f1(y,logp.argmax(1)),'->',macro_f1(y,adjusted))
print('Mean fold gain:',fold_results.delta.mean(),'wins:',int((fold_results.delta>0).sum()),'/5')


,fold,baseline,adjusted,delta,changed
0,1,0.981367,0.981626,0.000259,13
1,2,0.980392,0.979908,-0.000484,6
2,3,0.982930,0.982915,-0.000015,12
3,4,0.978785,0.979088,0.000303,18
4,5,0.980762,0.980380,-0.000381,10


,class0,class1,class2,class3,class4
1,0.0,-0.25,-0.05,0.0,-0.30
2,0.0,-0.10,-0.05,0.0,-0.15
3,0.0,-0.25,-0.05,0.0,-0.30
4,0.0,-0.35,-0.05,0.0,0.10
5,0.0,-0.35,-0.05,0.0,0.05


Pooled original F1: 0.9808489161304823 -> 0.9807807185336047
Mean fold gain: -6.357080384093994e-05 wins: 2 /5


## 5. Frozen development stress check — no bias fitting here

In [5]:
stress=[]
for fold in range(1,6):
    sample=pd.read_parquet(SOURCE/f'stress_sample_fold{fold}.parquet');truth=sample.label.to_numpy(int)
    for condition,(phase,_,_) in CONDITIONS.items():
        if phase!='development':continue
        paths=[SOURCE/'stress_cache'/f'fold{fold}'/m/f'{condition}.parquet' for m in ['candidate','sampler']]
        a,b=[pd.read_parquet(path) for path in paths]
        assert a.id.tolist()==b.id.tolist()==sample.id.tolist()
        p=(a[[f'p{i}' for i in range(5)]].to_numpy()+b[[f'p{i}' for i in range(5)]].to_numpy())/2
        lp=np.log(np.clip(p,1e-12,1))
        before=macro_f1(truth,lp.argmax(1));after=macro_f1(truth,(lp+biases[fold]).argmax(1))
        stress.append({'fold':fold,'condition':condition,'suite':'compound' if condition.startswith('compound_') else 'single','baseline':before,'adjusted':after,'delta':after-before})
stress_results=pd.DataFrame(stress)
display(stress_results.groupby('suite')[['baseline','adjusted','delta']].mean())
display(stress_results.groupby('condition').delta.mean().sort_values())


,baseline,adjusted,delta
suite,,,
compound,0.926588,0.923533,-0.003055
single,0.975847,0.975126,-0.000721


condition
compound_color_noise            -0.005133
compound_dark_color_noise       -0.004754
compound_bright_color           -0.003881
compound_rotation_color_noise   -0.003861
speckle_2                       -0.002822
speckle_1                       -0.002403
dark_1                          -0.001611
jpeg_2                          -0.001610
compound_rotation_dark          -0.001109
gaussian_2                      -0.000390
blur_2                          -0.000003
blur_1                           0.000004
gaussian_1                       0.000405
compound_dark_noise              0.000406
dark_2                           0.000415
jpeg_1                           0.000802
Name: delta, dtype: float64

## 6. Save experimental evidence and final fit separately

In [6]:
# Final fit uses all original OOF labels; its training score is not an evaluation.
final_bias=fit_bias(logp,y)
config={'grid':GRID.tolist(),'passes':PASSES,'penalty':PENALTY,'sources':sources,'reference_class':0}
OUT.mkdir(parents=True,exist_ok=True)
config_path=OUT/'config.json'
if config_path.exists():assert json.loads(config_path.read_text())==config,'Changed configuration: choose a new output directory'
else:config_path.write_text(json.dumps(config,indent=2))
fold_results.to_parquet(OUT/'crossfit_original.parquet',index=False)
stress_results.to_parquet(OUT/'crossfit_stress.parquet',index=False)
oof.assign(crossfit_prediction=adjusted).to_parquet(OUT/'crossfit_predictions.parquet',index=False)
result={'final_bias':final_bias.tolist(),'crossfit_biases':{str(k):v.tolist() for k,v in biases.items()},
        'original_mean_gain':float(fold_results.delta.mean()),'original_fold_wins':int((fold_results.delta>0).sum()),
        'stress_mean_gain':{k:float(v) for k,v in stress_results.groupby('suite').delta.mean().items()},
        'test_used':False,'automatically_approved_for_export':False,
        'limitation':'Exploratory bias cross-fitting on reused OOF; not fully nested base-model validation'}
(OUT/'bias_results.json').write_text(json.dumps(result,indent=2))
display(result)
print('Saved:',OUT,'| No Test inference or submission generated. Review gains and regressions before applying final_bias.')


{'final_bias': [0.0,
  -0.35000000000000003,
  -0.04999999999999999,
  0.0,
  0.04999999999999999],
 'crossfit_biases': {'1': [0.0,
   -0.25,
   -0.04999999999999999,
   0.0,
   -0.30000000000000004],
  '2': [0.0,
   -0.09999999999999998,
   -0.04999999999999999,
   0.0,
   -0.15000000000000002],
  '3': [0.0, -0.25, -0.04999999999999999, 0.0, -0.30000000000000004],
  '4': [0.0,
   -0.35000000000000003,
   -0.04999999999999999,
   0.0,
   0.09999999999999998],
  '5': [0.0,
   -0.35000000000000003,
   -0.04999999999999999,
   0.0,
   0.04999999999999999]},
 'original_mean_gain': -6.357080384093994e-05,
 'original_fold_wins': 2,
 'stress_mean_gain': {'compound': -0.003055293517146514,
  'single': -0.0007214098996389784},
 'test_used': False,
 'automatically_approved_for_export': False,
 'limitation': 'Exploratory bias cross-fitting on reused OOF; not fully nested base-model validation'}

Saved: /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task4/models/results/class_bias_v1 | No Test inference or submission generated. Review gains and regressions before applying final_bias.
